In [34]:
import pandas as pd
import numpy as np
import re

In [35]:
df = pd.read_csv('data/raw/SZKO_3566_CTAB_20260424233527.csv', sep=';')

In [36]:
df = df.iloc[:, 1:-1]

df.columns.values[0] = "region"

In [37]:
display(df.head())

,region,absolwenci;uczelnie publiczne;ogółem;ogółem;ogółem;2014;[osoba],absolwenci;uczelnie publiczne;ogółem;ogółem;ogółem;2015;[osoba],absolwenci;uczelnie publiczne;ogółem;ogółem;ogółem;2016;[osoba],absolwenci;uczelnie publiczne;ogółem;ogółem;ogółem;2017;[osoba],absolwenci;uczelnie publiczne;ogółem;ogółem;ogółem;2018;[osoba],absolwenci;uczelnie publiczne;ogółem;ogółem;ogółem;2019;[osoba],absolwenci;uczelnie publiczne;ogółem;ogółem;ogółem;2020;[osoba],absolwenci;uczelnie publiczne;ogółem;ogółem;ogółem;2021;[osoba],absolwenci;uczelnie publiczne;ogółem;ogółem;ogółem;2022;[osoba],...,absolwenci;uczelnie niepubliczne;ogółem;ogółem;ogółem;2015;[osoba],absolwenci;uczelnie niepubliczne;ogółem;ogółem;ogółem;2016;[osoba],absolwenci;uczelnie niepubliczne;ogółem;ogółem;ogółem;2017;[osoba],absolwenci;uczelnie niepubliczne;ogółem;ogółem;ogółem;2018;[osoba],absolwenci;uczelnie niepubliczne;ogółem;ogółem;ogółem;2019;[osoba],absolwenci;uczelnie niepubliczne;ogółem;ogółem;ogółem;2020;[osoba],absolwenci;uczelnie niepubliczne;ogółem;ogółem;ogółem;2021;[osoba],absolwenci;uczelnie niepubliczne;ogółem;ogółem;ogółem;2022;[osoba],absolwenci;uczelnie niepubliczne;ogółem;ogółem;ogółem;2023;[osoba],absolwenci;uczelnie niepubliczne;ogółem;ogółem;ogółem;2024;[osoba]
0,DOLNOŚLĄSKIE,NaN,25961,25616,26636,23772,24157,22304,21764,20698,...,8756,7842,8160,6492,7079,6587,7963,8317,9662,9590
1,KUJAWSKO-POMORSKIE,NaN,12578,11352,11437,9649,9457,8104,8185,7799,...,5012,4859,5648,4588,4472,4328,4510,4276,5154,5089
2,LUBELSKIE,NaN,15294,14607,15374,13953,12936,12187,12304,12326,...,7406,6050,5831,4955,5308,4934,5143,5349,5350,5867
3,LUBUSKIE,NaN,4569,4130,3785,3020,2992,2932,2718,2709,...,252,122,159,156,97,148,130,173,176,156
4,ŁÓDZKIE,NaN,16539,14674,16351,14176,13219,11813,12504,11453,...,5366,4541,4578,4364,3957,3477,3537,4303,4857,4891


In [38]:
print(df.columns)

Index(['region',
       'absolwenci;uczelnie publiczne;ogółem;ogółem;ogółem;2014;[osoba]',
       'absolwenci;uczelnie publiczne;ogółem;ogółem;ogółem;2015;[osoba]',
       'absolwenci;uczelnie publiczne;ogółem;ogółem;ogółem;2016;[osoba]',
       'absolwenci;uczelnie publiczne;ogółem;ogółem;ogółem;2017;[osoba]',
       'absolwenci;uczelnie publiczne;ogółem;ogółem;ogółem;2018;[osoba]',
       'absolwenci;uczelnie publiczne;ogółem;ogółem;ogółem;2019;[osoba]',
       'absolwenci;uczelnie publiczne;ogółem;ogółem;ogółem;2020;[osoba]',
       'absolwenci;uczelnie publiczne;ogółem;ogółem;ogółem;2021;[osoba]',
       'absolwenci;uczelnie publiczne;ogółem;ogółem;ogółem;2022;[osoba]',
       'absolwenci;uczelnie publiczne;ogółem;ogółem;ogółem;2023;[osoba]',
       'absolwenci;uczelnie publiczne;ogółem;ogółem;ogółem;2024;[osoba]',
       'absolwenci;uczelnie niepubliczne;ogółem;ogółem;ogółem;2014;[osoba]',
       'absolwenci;uczelnie niepubliczne;ogółem;ogółem;ogółem;2015;[osoba]',
       'absolwe

In [39]:
import pandas as pd
import re

id_vars = ["region"]

value_vars = [c for c in df.columns if c not in id_vars]

df_long = df.melt(
    id_vars=id_vars,
    value_vars=value_vars,
    var_name="raw_col",
    value_name="absolwenci"
)

def parse_col(col):
    # jednostka
    unit = re.search(r"\[(.+?)\]", col)
    unit = unit.group(1) if unit else None
    
    # usuń jednostkę
    col = re.sub(r"\[.+?\]", "", col)
    
    parts = col.split(";")
    
    # rok
    year = next((p for p in parts if re.fullmatch(r"\d{4}", p)), None)
    
    # typ uczelni (drugi segment po "absolwenci")
    # np. absolwenci;uczelnie publiczne;...
    typ_uczelni = None
    if len(parts) > 1:
        typ_uczelni = parts[1]
    
    return pd.Series({
        "typ_uczelni": typ_uczelni,
        "rok": int(year) if year else None,
        "jednostka": unit
    })

df_long = pd.concat(
    [df_long, df_long["raw_col"].apply(parse_col)],
    axis=1
)

df_long = df_long.drop(columns=["raw_col", "jednostka"])

In [40]:
display(df_long)

,region,absolwenci,typ_uczelni,rok
0,DOLNOŚLĄSKIE,NaN,uczelnie publiczne,2014
1,KUJAWSKO-POMORSKIE,NaN,uczelnie publiczne,2014
2,LUBELSKIE,NaN,uczelnie publiczne,2014
3,LUBUSKIE,NaN,uczelnie publiczne,2014
4,ŁÓDZKIE,NaN,uczelnie publiczne,2014
...,...,...,...,...
347,ŚLĄSKIE,11166.0,uczelnie niepubliczne,2024
348,ŚWIĘTOKRZYSKIE,1378.0,uczelnie niepubliczne,2024
349,WARMIŃSKO-MAZURSKIE,1338.0,uczelnie niepubliczne,2024
350,WIELKOPOLSKIE,8391.0,uczelnie niepubliczne,2024


In [41]:
df_long.to_csv("data/processed/absolwenci.csv", index=False)

In [42]:
df_long.describe()

,absolwenci,rok
count,320.000000,352.000000
mean,10167.915625,2019.000000
std,9754.330755,3.166779
min,97.000000,2014.000000
25%,3702.750000,2016.000000
50%,6632.000000,2019.000000
75%,12667.500000,2022.000000
max,48316.000000,2024.000000


In [ ]:
df_long.missing_values = df_long.isnull().sum()
display(df_long.missing_values)

C:\Users\user\AppData\Local\Temp\ipykernel_24860\278907725.py:1: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  df_long.missing_values = df_long.isnull().sum()


region          0
absolwenci     32
typ_uczelni     0
rok             0
dtype: int64

: 